# 03 · Strategy backtest

**Purpose:** measure whether trading ahead of month-end 60/40 rebalancing makes money, using
the Calendar signal checked in notebook 02.

**Strategy** (`src/rebalancing_frontrun/backtest/strategy.py`)
- Active only in the last 5 trading days of each month.
- Position for each day is set at the **previous close** (no look-ahead).
- Signal > 0 (equities overweight) → short ES, long ZN; signal < 0 → the reverse. Sign only.
- Each leg sized to 10% annualized vol using a trailing 60-day estimate.
- Costs: per side on notional traded. Base case ≈ one tick: ES 0.5 bp, ZN 1.5 bp.

**Samples**
- In-sample: 2015-11 to 2023-03-17 (overlaps the end of the paper's sample)
- Out-of-sample: 2023-03-20 onward (after the paper's data ends)

**Pre-specified checks** (decided before seeing results, not tuning)
1. Paper timing: signal on each of the last 5 days sets the *next* day's position.
2. Cost sensitivity: zero, half-tick, one-tick.

**Inputs:** `data/processed/returns.parquet`

## Setup

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

from rebalancing_frontrun.backtest.strategy import run_backtest, summarize
from rebalancing_frontrun.data.databento_loader import REPO_ROOT

returns = pd.read_parquet(REPO_ROOT / "data" / "processed" / "returns.parquet")

IS_END, OOS_START = "2023-03-17", "2023-03-20"
SAMPLES = {"full": (None, None), "in-sample": (None, IS_END), "out-of-sample": (OOS_START, None)}

bt = run_backtest(returns)  # base case
print(f"{bt['active'].sum()} active days across "
      f"{bt[bt['active']].index.to_period('M').nunique()} months")

650 active days across 130 months


## 1 · Base case results

Daily stats count every trading day (flat days are 0). Each month's 5-day window is one trade.
P&L is additive: cumulative P&L is a running sum.

In [2]:
pd.concat({name: summarize(bt, s, e) for name, (s, e) in SAMPLES.items()}, axis=1)

full         in-sample        out-of-sample       
                 gross     net     gross    net         gross    net
ann_return_%      2.53    1.44      4.01   2.84         -0.48  -1.41
ann_vol_%         7.53    7.53      8.19   8.19          5.97   5.99
sharpe            0.34    0.19      0.49   0.35         -0.08  -0.24
max_drawdown_%  -14.82  -16.50    -11.45 -12.59        -14.82 -16.50
trades          130.00  130.00     87.00  87.00         43.00  43.00
hit_rate_%       52.31   50.77     52.87  51.72         51.16  48.84
avg_trade_bps    21.27   12.09     33.76  23.88         -4.01 -11.71
t_stat_trades     1.17    0.67      1.41   1.00         -0.16  -0.46

## 2 · Cumulative P&L

In [3]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(bt.index, bt["gross"].cumsum() * 100, label="before costs")
ax.plot(bt.index, bt["net"].cumsum() * 100, label="after costs")
ax.axvline(pd.Timestamp(OOS_START), color="black", ls="--", lw=0.8)
ax.text(pd.Timestamp(OOS_START), ax.get_ylim()[1] * 0.9, " out-of-sample →", va="top")
ax.axhline(0, color="0.6", lw=0.6)
ax.set_ylabel("cumulative P&L (% of capital)")
ax.set_title("Calendar front-running strategy, base case")
ax.legend()
plt.tight_layout()
plt.show()

## 3 · P&L by trade (month)

Which months drive the result? A strategy that only works in a few crisis months is fragile.

In [4]:
month = bt.index.to_period("M")
trades = bt[bt["active"]].groupby(month[bt["active"]])["net"].sum() * 1e4  # bps

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.bar(trades.index.to_timestamp(), trades.values, width=20,
       color=["tab:green" if v > 0 else "tab:red" for v in trades.values])
ax.axvline(pd.Timestamp(OOS_START), color="black", ls="--", lw=0.8)
ax.set_ylabel("net P&L per trade (bps)")
ax.set_title("Monthly trades, after costs")
plt.tight_layout()
plt.show()

print("Best 5 trades (bps):\n", trades.nlargest(5).round(1).to_string())
print("\nWorst 5 trades (bps):\n", trades.nsmallest(5).round(1).to_string())

Best 5 trades (bps):
 trade_date
2016-06    844.2
2021-11    719.3
2020-10    604.2
2017-04    446.2
2018-05    422.3
Freq: M

Worst 5 trades (bps):
 trade_date
2017-11   -650.1
2020-02   -548.4
2019-05   -462.7
2021-03   -353.9
2025-02   -318.9
Freq: M


In [5]:
# How much of the full-sample result comes from 2020?
ex2020 = bt[bt.index.year != 2020]
print("Net Sharpe, full sample:", summarize(bt).loc["sharpe", "net"])
print("Net Sharpe, excluding 2020:", summarize(ex2020).loc["sharpe", "net"])

Net Sharpe, full sample: 0.19
Net Sharpe, excluding 2020: 0.2


## 4 · Check: paper timing

The paper uses the signal on each of the last 5 days to predict the **next** day's return. That
shifts our holding window by one day: the last 4 days of the month plus the first day of the
next month. This follows the paper's definition; it is not chosen from our results.

In [6]:
bt_paper = run_backtest(returns, timing="paper")
pd.concat(
    {f"{name} | {label}": summarize(b, s, e)["net"]
     for name, (s, e) in SAMPLES.items()
     for label, b in [("ours", bt), ("paper", bt_paper)]},
    axis=1,
)

,full | ours,full | paper,in-sample | ours,in-sample | paper,out-of-sample | ours,out-of-sample | paper
ann_return_%,1.44,-1.35,2.84,-1.37,-1.41,-1.29
ann_vol_%,7.53,7.53,8.19,7.95,5.99,6.61
sharpe,0.19,-0.18,0.35,-0.17,-0.24,-0.19
max_drawdown_%,-16.50,-29.60,-12.59,-29.12,-16.50,-12.42
trades,130.00,131.00,87.00,88.00,43.00,44.00
hit_rate_%,50.77,45.04,51.72,43.18,48.84,47.73
avg_trade_bps,12.09,-11.21,23.88,-11.43,-11.71,-10.52
t_stat_trades,0.67,-0.60,1.00,-0.46,-0.46,-0.40


## 5 · Check: cost sensitivity

One tick per side is conservative; a limit or passive fill often costs about half a tick.
Zero cost shows the pure signal.

In [7]:
cost_cases = {"zero": (0.0, 0.0), "half tick": (0.25, 0.75), "one tick (base)": (0.5, 1.5)}
rows = {}
for label, (es_c, zn_c) in cost_cases.items():
    b = run_backtest(returns, es_cost_bps=es_c, zn_cost_bps=zn_c)
    for name, (s, e) in SAMPLES.items():
        rows[(label, name)] = summarize(b, s, e).loc[["sharpe", "avg_trade_bps"], "net"]
pd.DataFrame(rows).T.unstack(1).round(2)

sharpe                         avg_trade_bps            \
                  full in-sample out-of-sample          full in-sample   
half tick         0.26      0.42         -0.16         16.68     28.82   
one tick (base)   0.19      0.35         -0.24         12.09     23.88   
zero              0.34      0.49         -0.08         21.27     33.76   

                               
                out-of-sample  
half tick               -7.86  
one tick (base)        -11.71  
zero                    -4.01

## Summary

Fill in after reviewing:
- [ ] Base case: in-sample vs out-of-sample Sharpe and t-stat (section 1)
- [ ] Is the result driven by a few months / 2020? (section 3)
- [ ] Does paper timing change the conclusion? (section 4)
- [ ] Does the result survive realistic costs? (section 5)

**Next:** placebo test (same trade on random 5-day windows) in notebook 04.